# Credit Card Transactions Analytics & Fraud Analysis

## Overview

This notebook performs data cleaning, preprocessing, and feature engineering on a large-scale credit card transactions dataset to create analysis-ready fact and dimension tables for downstream analytics and dashboard development.

## Dataset Source

Transactions Fraud Datasets (Kaggle)

https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets

## Objectives

- Clean and validate raw transaction, customer, card, and merchant datasets
- Handle missing values, data type inconsistencies, and data quality issues
- Engineer transaction, customer, and card-level analytical features
- Build a dimensional model consisting of:
  - `fact_transactions`
  - `dim_users`
  - `dim_cards`
  - `dim_mcc`
- Export cleaned datasets for Excel analysis and Power BI dashboard development

## Tools

Python • Pandas • NumPy • Jupyter Notebook • Excel • Power BI

## Loading Libraries and Raw Datasets

In [2]:
# Importing the required libraries to perform Data Cleaning
import pandas as pd
import numpy as np
import json

In [3]:
# Importing the raw datasets csv files into Pandas Dataframes
transactions_df = pd.read_csv('transactions_2019.csv')
cards_df = pd.read_csv('cards_data.csv')
users_df = pd.read_csv('users_data.csv')

# Reading the json file for fraud lables on transactions and converting it into dataframe
with open("train_fraud_labels.json", "r") as f:
    fraud_data = json.load(f)
fraud_df = pd.DataFrame(fraud_data)

# Reading json mile on mcc codes as a Pandas Dataframe
with open("mcc_codes.json", "r") as f:
    mcc = pd.DataFrame.from_dict(json.load(f), orient="index")



## Transactions Dataset Cleaning & Feature Engineering

### Transactions Dataset Summary

Dataset Size: 1,159,966 rows × 12 columns

Key Cleaning Steps:
- Converted transaction amount to numeric
- Joined with fraud dataset to add fraud lables for transactions
- Imputed missing fraud labels as "Unknown"
- Standardized categorical variables
- Validated negative transaction amounts

Key Features Created:
- Hour
- Day Part
- Amount Bucket
- Transaction Type
- Fraud Flag

In [4]:
# Inspecting the size of the transactions Dataframe
transactions_df.shape

(1159966, 12)

In [5]:
# Inspecting first 5 rows of the dataframe
transactions_df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,22326462,2019-01-01 00:02:00,496,3186,$119.35,Chip Transaction,30286,Mesquite,TX,75149.0,4814,NaN
1,22326465,2019-01-01 00:05:00,1129,2677,$100.00,Chip Transaction,27092,Vista,CA,92084.0,4829,NaN
2,22326466,2019-01-01 00:06:00,114,5283,$51.71,Chip Transaction,61195,North Hollywood,CA,91606.0,5541,NaN
3,22326467,2019-01-01 00:06:00,641,2774,$105.30,Swipe Transaction,75781,Columbus,OH,43228.0,5411,NaN
4,22326468,2019-01-01 00:10:00,114,5283,$82.00,Chip Transaction,61195,North Hollywood,CA,91606.0,5541,NaN


In [6]:
# Checking Datatypes of each column
transactions_df.dtypes

id                  int64
date               object
client_id           int64
card_id             int64
amount             object
use_chip           object
merchant_id         int64
merchant_city      object
merchant_state     object
zip               float64
mcc                 int64
errors             object
dtype: object

### Data Type Corrections 

In [7]:
# Converting the date feature to datetime 
transactions_df['date'] = pd.to_datetime(transactions_df['date'])

In [8]:
# Converting the amount feature into numeric 
transactions_df['amount'] = pd.to_numeric(
    transactions_df['amount'].str.replace('$', '', regex=False),
    errors='coerce'
)

### Rename id Column to transaction id

In [9]:
transactions_df.rename(columns ={'id':'transaction_id'},inplace=True)

### Checking Anomalies and Outliers in the amount column

In [10]:
# Inspecting the distribution of amount
transactions_df['amount'].describe()

count    1.159966e+06
mean     4.265216e+01
std      8.058645e+01
min     -5.000000e+02
25%      8.880000e+00
50%      2.886000e+01
75%      6.314000e+01
max      6.613440e+03
Name: amount, dtype: float64

### Comment on Distribution of amount
The minimum value of amount is negative which may suggest Refund transactions and needs to be further examined. The maximum value is 6.6k dollars which is not an extreme outlier as it may indicate an electronics or Jewellery purhchase.

In [13]:
# Checking the total rows with negative amount values
(transactions_df['amount'] < 0).sum()

56900

In [15]:
# Confirming if the negative transactions are majorly system errors
transactions_df.loc[transactions_df['amount'] < 0, 'errors'].value_counts(dropna =False)

errors
NaN                             56094
Insufficient Balance              584
Bad PIN                           111
Technical Glitch                  101
Bad Zipcode                         8
Bad PIN,Insufficient Balance        1
Bad Card Number                     1
Name: count, dtype: int64

### Observation
Since only **1.4%** of transactions with negative amount are flagged as error we can conclude that these transactions are the **Refund/Reversal** type of transactions and create a feature **Transaction type** to distinguish these transactions.

In [16]:
# Creating a Feature to distinguish Refund Transactions 
transactions_df['transaction_type'] = np.where(
    transactions_df['amount'] < 0,
    'Refund/Reversal',
    'Purchase'
)

### Handling Missing Values

In [17]:
# Checking number of missing values in each column
transactions_df.isna().sum()

transaction_id            0
date                      0
client_id                 0
card_id                   0
amount                    0
use_chip                  0
merchant_id               0
merchant_city             0
merchant_state       141071
zip                  150932
mcc                       0
errors              1141312
transaction_type          0
dtype: int64

### Observation
Since **98.39%** of all rows are null values in the error column we can safely mark these transactions as **No Error**. Further since the merchant city does not have any missing values we can keep the rows where merchant state and zip are missing and mark the null values as **Unknown**.

In [18]:
# Imputing missing values with default values
transactions_df['errors'] = (
    transactions_df['errors']
    .fillna('No Error')
)

transactions_df['zip'] = (
    transactions_df['zip']
    .fillna('Unknown')
)

transactions_df['merchant_state'] = (
    transactions_df['merchant_state']
    .fillna('Unknown')
)


### Removing Duplicate Records

In [20]:
# Checking Total duplicate rows in the dataframe
transactions_df.duplicated().sum()

0

### Observation
There are no duplicate rows in the transactions dataset.

### Standardization of Text Columns

In [24]:
# Creating a list of string columns excluding the zip column which has partial numeric data
string_cols = (
    transactions_df.select_dtypes(include='object')
    .columns
    .drop('zip')
)

In [27]:
# Removing leading and trailing whitespaces from text columns
for col in string_cols:
    transactions_df[col] = transactions_df[col].str.strip()

In [29]:
# Converting text columns to title case
title_case_cols = [
    'merchant_city',
    'errors',
    'transaction_type'
]

for col in title_case_cols:
    transactions_df[col] = transactions_df[col].str.title()

### Feature Engineering

In [30]:
# Creating hour feature to extract hour of transactions
transactions_df['hour'] = transactions_df['date'].dt.hour

# Creating Time of day feature based on hour column
transactions_df['time_of_day'] = np.select(
    [
        transactions_df['hour'].between(5, 11),
        transactions_df['hour'].between(12, 16),
        transactions_df['hour'].between(17, 20),
        (transactions_df['hour'] >= 21) | (transactions_df['hour'] <= 4)
    ],
    [
        'Morning',
        'Afternoon',
        'Evening',
        'Night'
    ],
    default='Unknown'
)

# Creating Amount Bucket feature to segment transactions in Small,Medium and Large Categories

transactions_df['amount_bucket'] = pd.cut(
    transactions_df['amount'],
    bins=[-np.inf, 50, 200, np.inf],
    labels=['Small', 'Medium', 'Large'],
    right=False
)


## Fraud Labels Dataset Cleaning

### Fraud label dataset summary
Dataset size : (8914963, 2)

Key Cleaning Steps:

- Moving transaction ids from index to column and renaming columns
- Converted transaction id into a numeric field
- Checked for duplicate rows and missing values


In [34]:
# Inspecting first 5 rows of Fraud Dataframe
fraud_df.head()

,target
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [35]:
# Resetting index and renaming columns for Fraud Dataframe
fraud_df.reset_index(inplace=True)
fraud_df.rename(columns ={'index':'transaction_id','target':'is_fraud'},inplace = True)

### Data Type Correction

In [38]:
# Inspecting Datatype of columns 
fraud_df.dtypes

transaction_id    object
is_fraud          object
dtype: object

In [39]:
# Converting the transaction id column to numeric
fraud_df['transaction_id'] = fraud_df['transaction_id'].astype(int)

### Checing Duplicate Rows and Missing Values

In [40]:
fraud_df.duplicated().sum()

0

In [41]:
fraud_df.isna().sum()

transaction_id    0
is_fraud          0
dtype: int64

### Observation
There are no missing values in the 2 columns and no duplicate rows in the fraud dataframe.

### Merging the Fraud label dataframe with transactions dataframe

The fraud lables are merged with the transactions dataset to categorize each transactions as fraud or legitimate.

In [45]:
transactions_df = transactions_df.merge(
    fraud_df,
    on ='transaction_id',
    how='left',
    validate ='one_to_one'
)

In [48]:
# Checking that no row multiplication occured due to merging
assert (
    len(transactions_df)
    ==
    transactions_df['transaction_id'].nunique()
)

In [47]:
# Inspecting Null Values after Merging
transactions_df.isna().sum()

transaction_id           0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state           0
zip                      0
mcc                      0
errors                   0
transaction_type         0
hour                     0
time_of_day              0
amount_bucket            0
is_fraud            382627
dtype: int64

### Fraud Label Missing Values

After merging the transaction and fraud datasets, approximately 33% of transactions had missing fraud labels.

To investigate, the total number of unique transactions in the source data (13.3 million) was compared with the number of transactions present in the fraud labels dataset (8.9 million). This confirmed that the fraud dataset contains labels for only a subset of all transactions.

Therefore, missing fraud labels were not assumed to represent legitimate transactions. Instead, they were assigned the value **"Unknown"** to preserve the distinction between confirmed legitimate transactions and transactions for which no fraud assessment was available.

In [49]:
#Imputing default values in missing fraud labels
transactions_df['is_fraud'] = (
    transactions_df['is_fraud']
    .fillna('Unknown')
)

In [51]:
# Creating Binary Fraud Flag based on Fraud label column
transactions_df['fraud_flag'] = (
    transactions_df['is_fraud']
    .map({
        'Yes': 1,
        'No': 0
    })
)

### Dropping the Zip Column from transactions dataset

Since the Zip column is a high cardinality column and is not relevant to the analyses planned in the project, this column was dropped to reduce data size and improve performance.

In [55]:
transactions_df = transactions_df.drop('zip',axis=1)

## Cards Dataset Cleaning & Feature Engineering

### Cards Dataset Summary

Dataset Size: 6146 rows × 13 columns

Key Cleaning Steps:
- Converted credit limit to numeric and acct open date to datetime
- Renamed id column to card_id
- Checked Missing Values and Duplicate rows 
- Standardized categorical variables
- Dropped columns like cvv and card number which are not useful in analysis

Key Features Created:
- Expiry Month and Expiry Year
- Card Age (in years)
- Chip Flag
- Card Age Group

In [59]:
# Inspecting first 5 rows of cards dataframe
cards_df.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


### Checking Missing Values and Duplicate Records

In [64]:
### Checking Missing Values and Duplicate Rows in Cards Dataframe
print('duplicate rows:',cards_df.duplicated().sum())
print("Missing Values in Columns")
print(cards_df.isna().sum())

duplicate rows: 0
Missing Values in Columns
id                       0
client_id                0
card_brand               0
card_type                0
card_number              0
expires                  0
cvv                      0
has_chip                 0
num_cards_issued         0
credit_limit             0
acct_open_date           0
year_pin_last_changed    0
card_on_dark_web         0
dtype: int64


### Observation
There are no missing values and duplicate records in the cards dataframe.

In [65]:
# Inspecting column card_on_dark_web to check relevance for analysis
cards_df['card_on_dark_web'].value_counts()

card_on_dark_web
No    6146
Name: count, dtype: int64

### Dropping Columns not relevant for Analysis

The columns like cvv and card number are personal information and are not useful for analyses planned in this project thus these columns are dropped. Further the column card_on_dark_web has value No in all rows of the dataset thus it is useless in analyis and is also dropped.

In [66]:
# Dropping the columns not required in Analysis
cards_df = cards_df.drop(
    columns=[
        'card_number',
        'cvv',
        'card_on_dark_web'
    ]
)

### Renaming id Column to card id

In [68]:
cards_df = cards_df.rename(columns={
    'id': 'card_id'
})

### Datatype Correction

In [69]:
# Inspecting Datatype for columns
cards_df.dtypes

card_id                   int64
client_id                 int64
card_brand               object
card_type                object
expires                  object
has_chip                 object
num_cards_issued          int64
credit_limit             object
acct_open_date           object
year_pin_last_changed     int64
dtype: object

In [70]:
# Converting the credit limit column to numeric
cards_df['credit_limit'] = (
    cards_df['credit_limit']
    .str.replace('$', '', regex=False)
    .astype(float)
)

In [71]:
# Conerting the acct open date to a datetime column
cards_df['acct_open_date'] = pd.to_datetime(
    cards_df['acct_open_date'],
    format='%m/%Y'
)

### Feature Engineering

In [72]:
# Creating the expiry month and expiry year columns from expires
cards_df['expiry_month'] = (
    cards_df['expires']
    .str.split('/')
    .str[0]
    .astype(int)
)

cards_df['expiry_year'] = (
    cards_df['expires']
    .str.split('/')
    .str[1]
    .astype(int)
)

In [73]:
# Dropping the original expires column 
cards_df = cards_df.drop(columns=['expires'])

In [74]:
# Creating Card age in years feature based on dataset enddate
analysis_date = pd.Timestamp('2019-12-31')

cards_df['card_age_years'] = (
    (analysis_date - cards_df['acct_open_date'])
    .dt.days / 365.25
).round(1)

In [75]:
# Checking Distribution of card age
cards_df['card_age_years'].describe()

count    6146.000000
mean        8.952083
std         6.113270
min        -0.100000
25%         3.700000
50%         9.850000
75%        13.200000
max        29.000000
Name: card_age_years, dtype: float64

In [76]:
# Limiting the minimum card age to zero 
cards_df['card_age_years'] = cards_df['card_age_years'].clip(lower=0)

In [77]:
# Creating Card Age group feature
cards_df['card_age_group'] = pd.cut(
    cards_df['card_age_years'],
    bins=[0, 2, 5, 10, np.inf],
    labels=[
        'New (<2 yrs)',
        '2-5 Years',
        '5-10 Years',
        '10+ Years'
    ],
    include_lowest=True
)

In [78]:
# Creating chip flag column 
cards_df['chip_flag'] = (
    cards_df['has_chip']
    .str.upper()
    .map({
        'YES': 1,
        'NO': 0
    })
)

### Standardization of Text Columns

In [79]:
# Removing whitespaces and converting to title case for text columns
text_cols = [
    'card_brand',
    'card_type'
]

for col in text_cols:
    cards_df[col] = (
        cards_df[col]
        .str.strip()
        .str.title()
    )

## Users Dataset Cleaning & Feature Engineering

### Users Dataset Summary

Dataset Size: 2000 rows × 14 columns

Key Cleaning Steps:
- Converted income columns to numeric
- Renamed id column to client_id
- Checked Missing Values and Duplicate rows 
- Dropped Address Column not useful in analysis

Key Features Created:
- Age Group
- Income Group
- Credit Score Group
- Debt to Income ratio

In [81]:
# Inspecting the first 5 rows of users dataframe
users_df.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


### Dropping Address Column

In [82]:
# Dropping address column not required in analysis
users_df = users_df.drop(columns=['address'])

### Renaming id column to client id

In [83]:
users_df = users_df.rename(columns={'id': 'client_id'})

### Checking Missing Values and Duplicate Records

In [84]:
### Checking Missing Values and Duplicate Rows in users Dataframe
print('duplicate rows:',users_df.duplicated().sum())
print("Missing Values in Columns")
print(users_df.isna().sum())

duplicate rows: 0
Missing Values in Columns
client_id            0
current_age          0
retirement_age       0
birth_year           0
birth_month          0
gender               0
latitude             0
longitude            0
per_capita_income    0
yearly_income        0
total_debt           0
credit_score         0
num_credit_cards     0
dtype: int64


### Observation

There are no missing values or duplicate rows in the users dataframe.

### Datatype Correction

In [85]:
# Converting income columns into numeric type
money_cols = [
    'per_capita_income',
    'yearly_income',
    'total_debt'
]

for col in money_cols:
    users_df[col] = pd.to_numeric(
        users_df[col]
        .str.replace('$', '', regex=False),
        errors='coerce'
    )


In [86]:
# Checking distribution of the income columns
users_df[money_cols].describe()

,per_capita_income,yearly_income,total_debt
count,2000.000000,2000.000000,2000.000000
mean,23141.928000,45715.882000,63709.694000
std,11324.137358,22992.615456,52254.453421
min,0.000000,1.000000,0.000000
25%,16824.500000,32818.500000,23986.750000
50%,20581.000000,40744.500000,58251.000000
75%,26286.000000,52698.500000,89070.500000
max,163145.000000,307018.000000,516263.000000


### Observation
The minimum and maximum values of the income columns do not indicate presense of extreme outliers and all values appear to be realistic.

### Feature Engineering

In [87]:
# Creating Age Group feature
users_df['age_group'] = pd.cut(
    users_df['current_age'],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-55',
        '56-65',
        '65+'
    ],
    include_lowest=True
)

In [88]:
# Checking Distribution of Age Groups
users_df['age_group'].value_counts()

age_group
46-55    385
26-35    357
36-45    342
18-25    337
65+      291
56-65    288
Name: count, dtype: int64

In [89]:
# Creating Income group Feature
users_df['income_group'] = pd.cut(
    users_df['yearly_income'],
    bins=[0, 30000, 45000, 60000, np.inf],
    labels=[
        'Low Income',
        'Lower Middle',
        'Upper Middle',
        'High Income'
    ],
    include_lowest=True
)

In [90]:
# Creating credit score group feature
users_df['credit_score_group'] = pd.cut(
    users_df['credit_score'],
    bins=[0, 580, 670, 740, 800, 900],
    labels=[
        'Poor',
        'Fair',
        'Good',
        'Very Good',
        'Excellent'
    ]
)

In [93]:
# Creating debt to income ratio feature
users_df['debt_to_income_ratio'] = (
    users_df['total_debt']
    / users_df['yearly_income']
).round(3)

## Cleaning the MCC dataframe

MCC dataset summary : 109 rows x 2 columns

Key cleaning steps:
- Resetting index and renaming columns
- Checking missing values and duplicate records
- Standardization of Mcc descriptions
- Data Type Correction

In [95]:
# Inspecting first 5 rows of mcc dataframe
mcc.head()

,0
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees


### Renaming columns of MCC dataframe

In [96]:
# Resetting index and renaming columns of mcc dataframe
mcc.reset_index(inplace=True)
mcc = mcc.rename(columns={
    'index': 'mcc',
     0 : 'mcc_description'
     })

### Checking Missing Values and Duplicate Records in MCC Dataframe

In [98]:
# Checking missing values
mcc.isna().sum()

mcc                0
mcc_description    0
dtype: int64

In [99]:
# Checking duplicate records
mcc.duplicated().sum()

0

### Observation
Thre are no missing values or duplicate records in mcc dataframe.

### Standardization of MCC Description Column

In [100]:
mcc['mcc_description'] = (
    mcc['mcc_description']
    .str.strip()
    .str.title()
)

### Datatype Correction

In [101]:
# Checkingg Datatype of columns
mcc.dtypes

mcc                object
mcc_description    object
dtype: object

In [102]:
# Converting the mcc codes into numeric column
mcc['mcc'] = mcc['mcc'].astype(int)

## Data Quality Validation of All Dataframes
Before exporting the cleaned datasets it is best practice to validate that there are no missing values and datatypes of columns are correct.

### Validating Transactions Dataframe

In [104]:
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1159966 entries, 0 to 1159965
Data columns (total 17 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   transaction_id    1159966 non-null  int64         
 1   date              1159966 non-null  datetime64[ns]
 2   client_id         1159966 non-null  int64         
 3   card_id           1159966 non-null  int64         
 4   amount            1159966 non-null  float64       
 5   use_chip          1159966 non-null  object        
 6   merchant_id       1159966 non-null  int64         
 7   merchant_city     1159966 non-null  object        
 8   merchant_state    1159966 non-null  object        
 9   mcc               1159966 non-null  int64         
 10  errors            1159966 non-null  object        
 11  transaction_type  1159966 non-null  object        
 12  hour              1159966 non-null  int32         
 13  time_of_day       1159966 non-null  object

### Validating Cards Dataframe

In [105]:
cards_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   card_id                6146 non-null   int64         
 1   client_id              6146 non-null   int64         
 2   card_brand             6146 non-null   object        
 3   card_type              6146 non-null   object        
 4   has_chip               6146 non-null   object        
 5   num_cards_issued       6146 non-null   int64         
 6   credit_limit           6146 non-null   float64       
 7   acct_open_date         6146 non-null   datetime64[ns]
 8   year_pin_last_changed  6146 non-null   int64         
 9   expiry_month           6146 non-null   int64         
 10  expiry_year            6146 non-null   int64         
 11  card_age_years         6146 non-null   float64       
 12  card_age_group         6146 non-null   category      
 13  chi

### Validating Users Dataframe

In [106]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   client_id             2000 non-null   int64   
 1   current_age           2000 non-null   int64   
 2   retirement_age        2000 non-null   int64   
 3   birth_year            2000 non-null   int64   
 4   birth_month           2000 non-null   int64   
 5   gender                2000 non-null   object  
 6   latitude              2000 non-null   float64 
 7   longitude             2000 non-null   float64 
 8   per_capita_income     2000 non-null   int64   
 9   yearly_income         2000 non-null   int64   
 10  total_debt            2000 non-null   int64   
 11  credit_score          2000 non-null   int64   
 12  num_credit_cards      2000 non-null   int64   
 13  age_group             2000 non-null   category
 14  income_group          2000 non-null   category
 15  cred

### Validating MCC Dataframe

In [107]:
mcc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   mcc              109 non-null    int64 
 1   mcc_description  109 non-null    object
dtypes: int64(1), object(1)
memory usage: 1.8+ KB


### Final Validation Summary

- Data quality checks confirmed that all four datasets contain no missing values across their respective columns, with the exception of the `fraud_flag` column in the transactions dataset. Null values in this column were intentionally retained to distinguish transactions with no available fraud assessment from transactions confirmed as legitimate.

- Data types were reviewed and validated to ensure consistency with the underlying data and intended analytical use.

- Dataset dimensions (row and column counts) were verified against expected values following all cleaning, transformation, and feature engineering steps.

- Based on the above validations, the fact and dimension tables are considered analysis-ready and can be exported for downstream Excel analysis, Power BI data modeling, and dashboard development.

### Exporting Cleaned Datasets for Analysis

In [108]:
transactions_df.to_csv(
    'fact_transactions.csv',
    index=False
)

cards_df.to_csv(
    'dim_cards.csv',
    index=False
)

users_df.to_csv(
    'dim_users.csv',
    index=False
)

mcc.to_csv(
    'dim_mcc.csv',
    index=False
)